In [1]:
import pandas as pd
import joblib
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. تحميل البيانات وتجهيزها
df_ar = pd.read_csv('../data/arabic_reviews.csv') 
df_en = pd.read_csv('../data/imdb_reviews.csv')

ar_texts = df_ar['review_description'].dropna().astype(str).tolist()[:10000]
en_texts = df_en['review'].dropna().astype(str).tolist()[:10000]

X = ar_texts + en_texts
y = ['Arabic'] * len(ar_texts) + ['English'] * len(en_texts)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. تعريف الخوارزميات المراد مقارنتها
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Linear SVM': LinearSVC()
}

best_model = None
best_accuracy = 0.0
best_model_name = ""

# 3. تدريب ومقارنة كل خوارزمية
for name, clf in classifiers.items():
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=10000, analyzer='char_wb')),
        ('clf', clf)
    ])
    
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"=== {name} ===")
    print(f"Training Time: {train_time:.4f} sec")
    print(f"Accuracy: {acc * 100:.2f}%\n")
    
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = pipeline
        best_model_name = name

# 4. حفظ النموذج الأفضل دقة
print(f"Best Classifier: {best_model_name} ({best_accuracy * 100:.2f}%)")
joblib.dump(best_model, 'Language_classifier_weights.pkl')
print("Best model saved to Language_classifier_weights.pkl")

=== Logistic Regression ===
Training Time: 7.0978 sec
Accuracy: 99.80%

=== Multinomial Naive Bayes ===
Training Time: 6.5052 sec
Accuracy: 98.78%

=== Linear SVM ===
Training Time: 6.6018 sec
Accuracy: 99.90%

Best Classifier: Linear SVM (99.90%)
Best model saved to Language_classifier_weights.pkl
